# Prac 08.1

In this homework you are going to build your first classifier for the CIFAR-10 dataset. This dataset contains 10 different classes and you can learn more about it [here](https://www.cs.toronto.edu/~kriz/cifar.html). This homework consists of the following tasks:
* Dataset inspection
* Building the network
* Training
* Evaluation

At the end, as usual, there will be a couple of questions for you to answer :-)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D
from tensorflow.keras import Model
from time import time

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

# Set the seeds for reproducibility
from numpy.random import seed
from tensorflow.random import set_seed
seed_value = 1234578790
seed(seed_value)
set_seed(seed_value)

### Step 0: Dataset Inspection

Load the dataset and make a quick inspection.

In [ ]:
# Load the dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
# Mapping from class ID to class name
classes = {0:'plane', 1:'car', 2:'bird', 3:'cat', 4:'deer',
           5:'dog', 6:'frog', 7:'horse', 8:'ship', 9:'truck'}

# Dataset params
num_classes = len(classes)
size = x_train.shape[0]

# Visualize random samples (as a plot with 3x6 samples)
for ii in range(18):
    plt.subplot(3,6,ii+1)
    # Pick a random sample
    idx = np.random.randint(0, size)
    # Show the image and the label
    plt.imshow(x_train[idx, ...])
    plt.title(classes[int(y_train[idx])])

Compute the class histogram (you can visualize it if you want). Is the dataset balanced?
> Absolutely

Hint: You might find [Counter](https://docs.python.org/3/library/collections.html#collections.Counter) tool useful. In any case, it's up to you how you compute the histogram.

In [ ]:
# Compute the class histogram
from collections import Counter
hist = Counter(y_train.flatten())

plt.bar(hist.keys(), hist.values()), plt.grid(True)
plt.xlabel('Traffic Sign ID'), plt.ylabel('Counts')

### Step 1: Data Preparation

In this step, you'll need to prepare the data for training, i.e., you will have to normalize it and encode the labels as one-hot vectors.

In [ ]:
from sklearn.model_selection import train_test_split

# Normalization
x_train = x_train / 255
X_test = x_test / 255

X_train, X_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print('Train set:   ', len(y_train), 'samples')
print('Val set:    ', len(y_val), 'samples')
print('Test set:    ', len(y_test), 'samples')

print('Sample dims: ', x_train.shape)

### Step 2: Building the Classifier

Build the CNN for CIFAR10 classification. For starters, you can use the same network we used in the lesson for the MNIST problem.
>I slightly modified the model :)

In [ ]:
inputs = Input(shape=(32, 32, 3))

conv1 = Conv2D(16, kernel_size=(3, 3), activation="relu")(inputs)
max_pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

conv2 = Conv2D(32, kernel_size=(3, 3), activation="relu")(max_pool1)
max_pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

flat = Flatten()(max_pool2)
dense1 = Dense(128, activation='relu')(flat)
outputs = Dense(10, activation='softmax')(dense1)

model = Model(inputs, outputs)

# Show the model
model.summary()

### Step 3: Training

Compile the model and train it.

In [ ]:
from tensorflow.keras.optimizers import Adam

epochs = 25
batch_size = 128

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    verbose=1
)

In [ ]:
# Show training history (this cell is complete, nothing to implement here :-) )
h = history.history
epochs = range(len(h['loss']))

plt.subplot(121), plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
plt.legend(['Train', 'Validation'])
plt.subplot(122), plt.plot(epochs, h['accuracy'], '.-',
                           epochs, h['val_accuracy'], '.-')
plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'])

print('Train Acc     ', h['accuracy'][-1])
print('Validation Acc', h['val_accuracy'][-1])

### Step 4: Evaluation

In this step, you have to calculate the accuracies and visualize some random samples. For the evaluation, you are going to use the test split from the dataset.

In [ ]:
y_true = y_test.flatten()
y_pred = np.argmax(model.predict(x_test), axis=1)

In [ ]:
# Compute and print the accuracy for each class
for class_id, class_name in classes.items():
    indices = np.where(y_true == class_id)[0]
    acc_ratio = np.sum(y_pred[indices] == y_true[indices]) / len(indices)
    print(class_name, acc_ratio)

In [ ]:
# Print the overall stats
ev = model.evaluate(X_test, y_test)
print('Test loss  ', ev[0])
print('Test metric', ev[1])

In [ ]:
# Show random samples
for ii in range(15):
    # Pick a random sample
    idx = np.random.randint(0, len(x_test))
    # Show the results
    plt.subplot(3,5,ii+1), plt.imshow(x_test[idx, ...])
    plt.title('True: ' + str(classes[y_true[idx]]) + ' | Pred: ' + str(classes[y_pred[idx]]))
    plt.axis('off')

### Questions
* What is the overall accuracy of the classifier?
> train $\approx$ 0.829, valid $\approx$ 0.651, test $\approx$ 0.653. From these metrics we can say that the model is overfitting. We can see the same from the acc graph: at some epoch, the train acc started increasing higher than the val acc

* What modifications would you do in order to improve the classification accuracy?
> I would like to add Dropout or add more Conv layers

* Make **one** modification (that you think can help) and train the classifier again. Does the accuracy improve?

In [ ]:
from tensorflow.keras.layers import Dropout

inputs = Input(shape=(32, 32, 3))

conv1 = Conv2D(16, kernel_size=(3, 3), activation="relu")(inputs)
max_pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

conv2 = Conv2D(32, kernel_size=(3, 3), activation="relu")(max_pool1)
max_pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

flat = Flatten()(max_pool2)
dropout1 = Dropout(0.5)(flat)

dense1 = Dense(128, activation='relu')(dropout1)
dropout2 = Dropout(0.3)(dense1)

outputs = Dense(10, activation='softmax')(dropout2)

model = Model(inputs, outputs)
model.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam

epochs = 25
batch_size = 128

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    verbose=1
)

In [ ]:
h = history.history

print('Train Acc     ', h['accuracy'][-1])
print('Validation Acc', h['val_accuracy'][-1])

In [ ]:
# Print the overall stats
ev = model.evaluate(X_test, y_test)
print('Test loss  ', ev[0])
print('Test metric', ev[1])

As you can see, the model begins generalize pattern and learn them, rather than just memorizing them